# PUAD_eval_multi

MVTec-AD の複数カテゴリ（推奨: bottle, cable, capsule, pill, grid）について、
- EfficientAD の image-level / pixel-level 指標
- PUAD の image-level 指標
を一括評価する notebook です。

## 保存先
- `comparison/EfficientAD/<category>/pixel_metrics.json`
- `comparison/PUAD/<category>/metrics.json`
- `comparison/PUAD/<category>/anomaly_breakdown.json`
- `comparison/tables/puad_efficientad_multi_metrics.csv`


In [1]:
# 0. GPU 確認
!nvidia-smi


Mon Mar  9 12:44:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 1. Drive mount
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
# 2. 変数設定
import os

BASE_DIR = "/content/drive/MyDrive/industrial-anomaly-detection-app"
DATA_ROOT = f"{BASE_DIR}/data/MVTec-AD"
MODEL_ROOT = f"{BASE_DIR}/pretrained/PUAD/mvtec_ad_models"
COMPARISON_ROOT = f"{BASE_DIR}/comparison"

CATEGORIES = ["bottle", "cable", "capsule", "pill", "grid"]
SIZE = "m"
FEATURE_EXTRACTOR = "student"
PUAD_REPO_DIR = "/content/PUAD"

for c in CATEGORIES:
    print(c, os.path.exists(os.path.join(DATA_ROOT, c)))


bottle True
cable True
capsule True
pill True
grid True


In [4]:
# 3. repo clone
if not os.path.exists(PUAD_REPO_DIR):
    !git clone https://github.com/LeapMind/PUAD.git /content/PUAD
else:
    print("PUAD repo already exists:", PUAD_REPO_DIR)


Cloning into '/content/PUAD'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 63 (delta 18), reused 35 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 844.05 KiB | 30.14 MiB/s, done.
Resolving deltas: 100% (18/18), done.


In [5]:
# 4. 依存関係
# Colab GPU + CUDA 版 torch を使う
!pip uninstall -y torch torchvision torchaudio || true
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q scikit-image scikit-learn scipy pandas matplotlib
print("Install done. 必要なら Runtime restart 後に次セルから再開してください。")


Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 129.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 109.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 82.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 67.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 92.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 741.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━

In [6]:
# 5. import 確認
import torch, pandas as pd, numpy as np, scipy, sklearn, skimage
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


torch: 2.5.1+cu121
cuda available: True
cuda version: 12.1
gpu: Tesla T4


In [7]:
# 6. Drive 上のファイル配置確認
import os

def verify_category_files(category):
    ds = os.path.join(DATA_ROOT, category)
    model_dir = os.path.join(MODEL_ROOT, "m_size", "MVTec-AD", category)
    legacy_model_dir = os.path.join(MODEL_ROOT, "m_size", "mvtec_anomaly_detection", category)
    teacher = os.path.join(MODEL_ROOT, "m_size", "teacher", "teacher.pt")

    print("\n==", category, "==")
    print("dataset:", ds, os.path.exists(ds))
    print(" train:", os.path.exists(os.path.join(ds, "train")))
    print(" test :", os.path.exists(os.path.join(ds, "test")))
    print(" gt   :", os.path.exists(os.path.join(ds, "ground_truth")))
    print("teacher:", teacher, os.path.exists(teacher))
    print("model_dir (preferred):", model_dir, os.path.exists(model_dir))
    print("model_dir (legacy)   :", legacy_model_dir, os.path.exists(legacy_model_dir))

for c in CATEGORIES:
    verify_category_files(c)



== bottle ==
dataset: /content/drive/MyDrive/industrial-anomaly-detection-app/data/MVTec-AD/bottle True
 train: True
 test : True
 gt   : True
teacher: /content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/PUAD/mvtec_ad_models/m_size/teacher/teacher.pt True
model_dir (preferred): /content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/PUAD/mvtec_ad_models/m_size/MVTec-AD/bottle True
model_dir (legacy)   : /content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/PUAD/mvtec_ad_models/m_size/mvtec_anomaly_detection/bottle True

== cable ==
dataset: /content/drive/MyDrive/industrial-anomaly-detection-app/data/MVTec-AD/cable True
 train: True
 test : True
 gt   : True
teacher: /content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/PUAD/mvtec_ad_models/m_size/teacher/teacher.pt True
model_dir (preferred): /content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/PUAD/mvtec_ad_models/m_size/MVTec-AD/cable True
model_dir (legacy)   : /con

In [8]:
# 7. path 準備
import sys
sys.path.append("/content/PUAD")

from puad.dataset import build_dataset
from puad.efficientad.inference import load_efficient_ad
from puad.puad import PUAD

print("imports ok")


imports ok


In [9]:
# 8. EfficientAD anomaly map helper と pixel 指標
import os
import json
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from skimage.measure import label, regionprops
from sklearn.metrics import roc_auc_score, average_precision_score

def save_json(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)
    print("saved:", path)

def efficientad_map(efficient_ad_inference, img: torch.Tensor):
    device = efficient_ad_inference.device
    img = img.to(device).unsqueeze(0)

    with torch.no_grad():
        teacher_output = efficient_ad_inference.teacher(img)
        student_output = efficient_ad_inference.student(img)
        autoencoder_output = efficient_ad_inference.autoencoder(img)

        normalized_teacher_output = efficient_ad_inference.normalization(teacher_output)
        student_output_st = student_output[:, : student_output.shape[1] // 2, :, :]
        student_output_ae = student_output[:, student_output.shape[1] // 2 :, :, :]

        diff_teacher_student = (normalized_teacher_output - student_output_st) ** 2
        diff_student_ae = (autoencoder_output - student_output_ae) ** 2

        student_map = torch.mean(diff_teacher_student, dim=1, keepdim=True)
        ae_map = torch.mean(diff_student_ae, dim=1, keepdim=True)

        resized_student_map = F.interpolate(student_map, size=(efficient_ad_inference.img_size, efficient_ad_inference.img_size), mode="bilinear", align_corners=False)
        resized_ae_map = F.interpolate(ae_map, size=(efficient_ad_inference.img_size, efficient_ad_inference.img_size), mode="bilinear", align_corners=False)

        resized_student_map = resized_student_map.squeeze()
        resized_ae_map = resized_ae_map.squeeze()

        resized_student_map = 0.1 * (resized_student_map - efficient_ad_inference.q_a_student) / (efficient_ad_inference.q_b_student - efficient_ad_inference.q_a_student)
        resized_ae_map = 0.1 * (resized_ae_map - efficient_ad_inference.q_a_autoencoder) / (efficient_ad_inference.q_b_autoencoder - efficient_ad_inference.q_a_autoencoder)

        anomaly_map = 0.5 * resized_student_map + 0.5 * resized_ae_map
        anomaly_score = torch.max(anomaly_map).item()

    return anomaly_score, anomaly_map.detach().cpu().numpy()

def load_mvtec_mask_from_test_path(img_path: str, img_size: int = 256):
    parts = img_path.split(os.sep)
    anomaly_type = parts[-2]
    filename = parts[-1]

    if anomaly_type == "good":
        return np.zeros((img_size, img_size), dtype=np.uint8)

    category_root = img_path.split(os.sep + "test" + os.sep)[0]
    mask_name = filename.replace(".png", "_mask.png")
    mask_path = os.path.join(category_root, "ground_truth", anomaly_type, mask_name)

    mask = Image.open(mask_path).convert("L")
    mask = mask.resize((img_size, img_size), Image.NEAREST)
    mask = np.array(mask)
    return (mask > 0).astype(np.uint8)

def min_max_norm(x):
    x = np.asarray(x)
    return (x - x.min()) / (x.max() - x.min() + 1e-12)

def compute_aupro(masks, preds, max_step=1000, expect_fpr=0.3):
    masks = np.asarray(masks, dtype=bool)
    preds = np.asarray(preds, dtype=np.float32)

    max_th = preds.max()
    min_th = preds.min()
    delta = (max_th - min_th) / max_step

    pros_mean = []
    fprs = []
    binary_score_maps = np.zeros_like(preds, dtype=bool)

    for step in range(max_step):
        th = max_th - step * delta
        binary_score_maps[preds > th] = True
        binary_score_maps[preds <= th] = False

        pro = []
        for i in range(len(binary_score_maps)):
            label_map = label(masks[i], connectivity=2)
            props = regionprops(label_map)
            for prop in props:
                x_min, y_min, x_max, y_max = prop.bbox
                cropped_pred = binary_score_maps[i][x_min:x_max, y_min:y_max]
                cropped_mask = prop.filled_image
                intersection = np.logical_and(cropped_pred, cropped_mask).astype(np.float32).sum()
                pro.append(intersection / prop.area)

        gt_masks_neg = ~masks
        fpr = np.logical_and(gt_masks_neg, binary_score_maps).sum() / (gt_masks_neg.sum() + 1e-12)
        pros_mean.append(np.mean(pro) if len(pro) > 0 else 0.0)
        fprs.append(fpr)

    pros_mean = np.array(pros_mean)
    fprs = np.array(fprs)
    idx = fprs <= expect_fpr
    fprs_selected = fprs[idx]
    pros_selected = pros_mean[idx]
    if len(fprs_selected) < 2:
        return np.nan
    fprs_selected = min_max_norm(fprs_selected)
    return np.trapz(pros_selected, fprs_selected)

def evaluate_efficientad_pixel(efficient_ad_inference, test_dataset):
    pixel_gt, pixel_pred, image_gt, image_pred = [], [], [], []

    for idx, (img, label) in enumerate(test_dataset):
        img_path, _ = test_dataset.samples[idx]
        gt_mask = load_mvtec_mask_from_test_path(img_path, img_size=efficient_ad_inference.img_size)
        score, amap = efficientad_map(efficient_ad_inference, img)
        pixel_gt.append(gt_mask)
        pixel_pred.append(amap)
        image_gt.append(0 if label == test_dataset.class_to_idx["good"] else 1)
        image_pred.append(score)

    pixel_gt = np.asarray(pixel_gt)
    pixel_pred = np.asarray(pixel_pred)
    image_gt = np.asarray(image_gt)
    image_pred = np.asarray(image_pred)

    return {
        "Image_AUROC": float(roc_auc_score(image_gt, image_pred)),
        "Pixel_AUROC": float(roc_auc_score(pixel_gt.ravel(), pixel_pred.ravel())),
        "Pixel_AP": float(average_precision_score(pixel_gt.ravel(), pixel_pred.ravel())),
        "AUPRO": float(compute_aupro(pixel_gt, pixel_pred)),
    }


In [10]:
# 9. モデル配置の互換コピー（legacy -> preferred）
import shutil

for category in CATEGORIES:
    preferred = os.path.join(MODEL_ROOT, "m_size", "MVTec-AD", category)
    legacy = os.path.join(MODEL_ROOT, "m_size", "mvtec_anomaly_detection", category)
    if os.path.exists(preferred):
        print("preferred exists:", preferred)
        continue
    if os.path.exists(legacy):
        os.makedirs(preferred, exist_ok=True)
        for name in ["autoencoder.pt", "student.pt", "mu.pt", "sigma.pt", "quantile.pt"]:
            src = os.path.join(legacy, name)
            dst = os.path.join(preferred, name)
            if os.path.exists(src) and not os.path.exists(dst):
                shutil.copy2(src, dst)
        print("copied legacy -> preferred:", category)
    else:
        print("model missing:", category)


preferred exists: /content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/PUAD/mvtec_ad_models/m_size/MVTec-AD/bottle
preferred exists: /content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/PUAD/mvtec_ad_models/m_size/MVTec-AD/cable
preferred exists: /content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/PUAD/mvtec_ad_models/m_size/MVTec-AD/capsule
preferred exists: /content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/PUAD/mvtec_ad_models/m_size/MVTec-AD/pill
preferred exists: /content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/PUAD/mvtec_ad_models/m_size/MVTec-AD/grid


In [11]:
# 10. 単カテゴリ評価
def evaluate_puad_category(category):
    dataset_path = os.path.join(DATA_ROOT, category)
    train_dataset, valid_dataset, test_dataset = build_dataset(dataset_path, img_size=256)

    efficient_ad_inference = load_efficient_ad(
        model_dir_path=MODEL_ROOT,
        size=SIZE,
        dataset_name="MVTec-AD",
        category=category,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )

    eff_metrics = evaluate_efficientad_pixel(efficient_ad_inference, test_dataset)
    save_json(f"{COMPARISON_ROOT}/EfficientAD/{category}/pixel_metrics.json", eff_metrics)

    puad = PUAD(feature_extractor=FEATURE_EXTRACTOR)
    puad.load_efficient_ad(efficient_ad_inference)
    puad.train(train_dataset)
    puad.valid(valid_dataset)

    puad_auroc, puad_auroc_for_anomalies = puad.auroc_for_anomalies(test_dataset)
    puad_metrics = {
        "Method": "PUAD",
        "Dataset": "MVTec-AD",
        "Category": category,
        "Image_AUROC": float(puad_auroc * 100),
        "feature_extractor": FEATURE_EXTRACTOR,
        "size": SIZE,
    }
    save_json(f"{COMPARISON_ROOT}/PUAD/{category}/metrics.json", puad_metrics)
    save_json(f"{COMPARISON_ROOT}/PUAD/{category}/anomaly_breakdown.json", puad_auroc_for_anomalies)

    return {
        "Category": category,
        "EfficientAD_Image_AUROC": eff_metrics["Image_AUROC"] * 100,
        "EfficientAD_Pixel_AUROC": eff_metrics["Pixel_AUROC"] * 100,
        "EfficientAD_Pixel_AP": eff_metrics["Pixel_AP"] * 100,
        "EfficientAD_AUPRO": eff_metrics["AUPRO"] * 100,
        "PUAD_Image_AUROC": puad_auroc * 100,
    }


In [12]:
# 11. 一括実行
all_results = []
for category in CATEGORIES:
    print("\n====================")
    print("CATEGORY:", category)
    print("====================")
    result = evaluate_puad_category(category)
    all_results.append(result)

all_results



CATEGORY: bottle


/content/PUAD/puad/efficientad/inference.py:138: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  teacher.load_state_dict(torch.load(os.path.join(model_dir_path, f"{size}_size"

saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/EfficientAD/bottle/pixel_metrics.json
saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/PUAD/bottle/metrics.json
saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/PUAD/bottle/anomaly_breakdown.json

CATEGORY: cable


/content/PUAD/puad/efficientad/inference.py:138: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  teacher.load_state_dict(torch.load(os.path.join(model_dir_path, f"{size}_size"

saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/EfficientAD/cable/pixel_metrics.json
saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/PUAD/cable/metrics.json
saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/PUAD/cable/anomaly_breakdown.json

CATEGORY: capsule


...

/content/PUAD/puad/efficientad/inference.py:138: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  teacher.load_state_dict(torch.load(os.path.join(model_dir_path, f"{size}_size"

saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/EfficientAD/pill/pixel_metrics.json
saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/PUAD/pill/metrics.json
saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/PUAD/pill/anomaly_breakdown.json

CATEGORY: grid


/content/PUAD/puad/efficientad/inference.py:138: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  teacher.load_state_dict(torch.load(os.path.join(model_dir_path, f"{size}_size"

saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/EfficientAD/grid/pixel_metrics.json
saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/PUAD/grid/metrics.json
saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/PUAD/grid/anomaly_breakdown.json


[{'Category': 'bottle',
  'EfficientAD_Image_AUROC': 100.0,
  'EfficientAD_Pixel_AUROC': 97.1624808298723,
  'EfficientAD_Pixel_AP': 55.720265497949704,
  'EfficientAD_AUPRO': 88.98994278006323,
  'PUAD_Image_AUROC': np.float64(99.92063492063492)},
 {'Category': 'cable',
  'EfficientAD_Image_AUROC': 97.24512743628186,
  'EfficientAD_Pixel_AUROC': 98.23094665660797,
  'EfficientAD_Pixel_AP': 66.03923053590431,
  'EfficientAD_AUPRO': 90.74294758150938,
  'PUAD_Image_AUROC': np.float64(97.76986506746627)},
 {'Category': 'capsule',
  'EfficientAD_Image_AUROC': 98.4443558037495,
  'EfficientAD_Pixel_AUROC': 98.60498975332114,
  'EfficientAD_Pixel_AP': 33.17429644481083,
  'EfficientAD_AUPRO': 91.91379552583088,
  'PUAD_Image_AUROC': np.float64(98.04547267650578)},
 {'Category': 'pill',
  'EfficientAD_Image_AUROC': 98.17239498090562,
  'EfficientAD_Pixel_AUROC': 97.75239242898597,
  'EfficientAD_Pixel_AP': 48.86794009898884,
  'EfficientAD_AUPRO': 89.80990190228424,
  'PUAD_Image_AUROC': np.

In [13]:
# 12. 集計 CSV
import pandas as pd
df = pd.DataFrame(all_results)
out_csv = f"{COMPARISON_ROOT}/tables/puad_efficientad_multi_metrics.csv"
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
df.to_csv(out_csv, index=False)
print("saved:", out_csv)
display(df)


saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/tables/puad_efficientad_multi_metrics.csv


,Category,EfficientAD_Image_AUROC,EfficientAD_Pixel_AUROC,EfficientAD_Pixel_AP,EfficientAD_AUPRO,PUAD_Image_AUROC
0,bottle,100.000000,97.162481,55.720265,88.989943,99.920635
1,cable,97.245127,98.230947,66.039231,90.742948,97.769865
2,capsule,98.444356,98.604990,33.174296,91.913796,98.045473
3,pill,98.172395,97.752392,48.867940,89.809902,98.008729
4,grid,100.000000,98.194952,24.613952,93.588967,100.000000


In [15]:
# EfficientAD 可視化画像保存セル
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

NUM_EFF_SAMPLES_PER_CATEGORY = 2

def efficientad_anomaly_map(efficient_ad_inference, img_tensor):
    """
    img_tensor: [C,H,W] normalized tensor
    returns:
      anomaly_score: float
      anomaly_map: [H,W] numpy
    """
    img = img_tensor.to(efficient_ad_inference.device).unsqueeze(0)

    with torch.no_grad():
        teacher_output = efficient_ad_inference.teacher(img)
        student_output = efficient_ad_inference.student(img)
        autoencoder_output = efficient_ad_inference.autoencoder(img)

        normalized_teacher_output = efficient_ad_inference.normalization(teacher_output)
        student_output_st = student_output[:, : student_output.shape[1] // 2, :, :]
        student_output_ae = student_output[:, student_output.shape[1] // 2 :, :, :]

        diff_teacher_student = (normalized_teacher_output - student_output_st) ** 2
        diff_student_ae = (autoencoder_output - student_output_ae) ** 2

        student_map = torch.mean(diff_teacher_student, dim=1, keepdim=True)
        ae_map = torch.mean(diff_student_ae, dim=1, keepdim=True)

        resized_student_map = F.interpolate(
            student_map,
            size=(efficient_ad_inference.img_size, efficient_ad_inference.img_size),
            mode="bilinear",
            align_corners=False,
        ).squeeze()

        resized_ae_map = F.interpolate(
            ae_map,
            size=(efficient_ad_inference.img_size, efficient_ad_inference.img_size),
            mode="bilinear",
            align_corners=False,
        ).squeeze()

        resized_student_map = 0.1 * (
            (resized_student_map - efficient_ad_inference.q_a_student)
            / (efficient_ad_inference.q_b_student - efficient_ad_inference.q_a_student)
        )
        resized_ae_map = 0.1 * (
            (resized_ae_map - efficient_ad_inference.q_a_autoencoder)
            / (efficient_ad_inference.q_b_autoencoder - efficient_ad_inference.q_a_autoencoder)
        )

        anomaly_map = 0.5 * resized_student_map + 0.5 * resized_ae_map
        anomaly_score = torch.max(anomaly_map).item()

        anomaly_map = anomaly_map.detach().cpu().numpy()

    return anomaly_score, anomaly_map


def denorm_imagenet(img_tensor):
    """
    img_tensor: normalized tensor [C,H,W]
    returns uint8 image [H,W,C]
    """
    mean = torch.tensor([0.485, 0.456, 0.406], device=img_tensor.device).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=img_tensor.device).view(3, 1, 1)
    x = img_tensor * std + mean
    x = torch.clamp(x, 0, 1)
    x = (x.permute(1, 2, 0).detach().cpu().numpy() * 255).astype(np.uint8)
    return x


def overlay_heatmap(image_uint8, anomaly_map):
    amap = anomaly_map.copy()
    amap = amap - amap.min()
    if amap.max() > 0:
        amap = amap / amap.max()
    return amap


eff_sample_rows = []

for category in CATEGORIES:
    print(f"\n===== EfficientAD visualization: {category} =====")

    dataset_path = os.path.join(DATA_ROOT, category)
    train_dataset, valid_dataset, test_dataset = build_dataset(dataset_path, img_size=256)

    efficient_ad_inference = load_efficient_ad(
        model_dir_path=MODEL_ROOT,
        size=SIZE,
        dataset_name="MVTec-AD",
        category=category,
        device="cuda",
    )

    dst_dir = f"{COMPARISON_ROOT}/EfficientAD/{category}/samples"
    os.makedirs(dst_dir, exist_ok=True)

    saved = 0
    idx_to_class = {i: c for c, i in test_dataset.class_to_idx.items()}

    # good を除いて anomaly を優先的に保存
    for idx in range(len(test_dataset)):
        img, label = test_dataset[idx]
        class_name = idx_to_class[label]

        if class_name == "good":
            continue

        score, amap = efficientad_anomaly_map(efficient_ad_inference, img)
        vis_img = denorm_imagenet(img)

        amap_norm = overlay_heatmap(vis_img, amap)

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(vis_img)
        axes[0].set_title(f"input ({class_name})")
        axes[0].axis("off")

        axes[1].imshow(vis_img)
        axes[1].imshow(amap_norm, cmap="jet", alpha=0.45)
        axes[1].set_title(f"overlay score={score:.4f}")
        axes[1].axis("off")

        axes[2].imshow(amap_norm, cmap="jet")
        axes[2].set_title("anomaly map")
        axes[2].axis("off")

        plt.tight_layout()
        save_path = os.path.join(dst_dir, f"{idx:03d}_{class_name}_score_{score:.4f}.png")
        plt.savefig(save_path, dpi=180, bbox_inches="tight")
        plt.close(fig)

        eff_sample_rows.append({
            "Category": category,
            "Index": idx,
            "AnomalyType": class_name,
            "Score": score,
            "Path": save_path,
        })

        saved += 1
        if saved >= NUM_EFF_SAMPLES_PER_CATEGORY:
            break

    print("saved:", saved, "->", dst_dir)


===== EfficientAD visualization: bottle =====


/content/PUAD/puad/efficientad/inference.py:138: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  teacher.load_state_dict(torch.load(os.path.join(model_dir_path, f"{size}_size"

saved: 2 -> /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/EfficientAD/bottle/samples

===== EfficientAD visualization: cable =====
saved: 2 -> /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/EfficientAD/cable/samples

===== EfficientAD visualization: capsule =====
saved: 2 -> /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/EfficientAD/capsule/samples

===== EfficientAD visualization: pill =====
saved: 2 -> /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/EfficientAD/pill/samples

===== EfficientAD visualization: grid =====
saved: 2 -> /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/EfficientAD/grid/samples


In [16]:
# EfficientAD 保存確認
for category in CATEGORIES:
    p = f"{COMPARISON_ROOT}/EfficientAD/{category}/samples"
    print("\n", category, "->", p, os.path.exists(p))
    if os.path.exists(p):
        print(os.listdir(p)[:5])


 bottle -> /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/EfficientAD/bottle/samples True
['000_broken_large_score_1.4279.png', '001_broken_large_score_2.6971.png']

 cable -> /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/EfficientAD/cable/samples True
['000_bent_wire_score_0.6386.png', '001_bent_wire_score_1.3788.png']

 capsule -> /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/EfficientAD/capsule/samples True
['000_crack_score_0.4557.png', '001_crack_score_3.5638.png']

 pill -> /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/EfficientAD/pill/samples True
['000_color_score_0.2514.png', '001_color_score_2.5432.png']

 grid -> /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/EfficientAD/grid/samples True
['000_bent_score_2.1464.png', '001_bent_score_2.3808.png']
